# Madrid Open Data Analysis Demo

This notebook explores the processed Madrid Open Data database in a simple, step-by-step manner:

1. **Connect to Database** - Access our processed Madrid data
2. **Explore All Tables** - See what data we have available
3. **Sample Data from Each Table** - Understanding the structure and content
4. **Simple Analysis Examples** - Basic insights from the data
5. **Visualization** - Maps and charts to understand the data

Let's start by exploring what's in our database!

## 1. Setup and Database Connection

In [84]:
# Import required libraries
import duckdb
import pandas as pd
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import plugins
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8')

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [85]:
# Connect to the processed Madrid Open Data database
DATABASE_PATH = '../rampa/duckdb/databases/rampa.db'
conn = duckdb.connect(DATABASE_PATH)

# Check available tables
tables = conn.execute("SHOW TABLES").fetchall()
print("Available tables in Madrid Open Data database:")
for table in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM {table[0]}").fetchone()[0]
    print(f"  📊 {table[0]}: {count:,} records")

print(f"\n✅ Connected to database with {len(tables)} tables")

Available tables in Madrid Open Data database:
  📊 dim_geography: 2,461 records
  📊 fact_demographics: 2,442 records
  📊 fact_demographics_age_groups: 148,190 records
  📊 fact_points_of_interest: 15,932 records
  📊 routing_infrastructure: 55,703 records

✅ Connected to database with 5 tables


## 2. Explore All Tables

Let's see what tables we have and look at sample data from each one:

In [86]:
# Let's explore each table one by one
print("=== EXPLORING ALL TABLES ===\n")

for table_name, in tables:
    print(f"📋 TABLE: {table_name.upper()}")
    print("=" * (len(table_name) + 12))
    
    try:
        # Get table structure
        columns = conn.execute(f"DESCRIBE {table_name}").fetchall()
        print(f"Columns ({len(columns)} total):")
        for col, dtype, *_ in columns:
            print(f"  • {col}: {dtype}")
        
        # Get record count
        count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        print(f"\nTotal records: {count:,}")
        
        # Show sample data if table has records
        if count > 0:
            print(f"\nSample data (first 3 rows):")
            sample_data = conn.execute(f"SELECT * FROM {table_name} LIMIT 3").fetchall()
            
            # Create a simple DataFrame to display nicely
            df_sample = pd.DataFrame(sample_data, columns=[col[0] for col in columns])
            
            # Truncate long text fields for display
            for col in df_sample.columns:
                if df_sample[col].dtype == 'object':
                    df_sample[col] = df_sample[col].astype(str).apply(
                        lambda x: x[:50] + '...' if len(str(x)) > 50 else x
                    )
            
            print(df_sample.to_string(index=False))
        else:
            print("\n⚠️ No data in this table")
            
    except Exception as e:
        print(f"❌ Error exploring table: {e}")
    
    print("\n" + "="*60 + "\n")

=== EXPLORING ALL TABLES ===

📋 TABLE: DIM_GEOGRAPHY
Columns (5 total):
  • district_id: VARCHAR
  • district_name: VARCHAR
  • neighborhood_name: VARCHAR
  • census_section_id: VARCHAR
  • geom: VARCHAR

Total records: 2,461

Sample data (first 3 rows):
district_id district_name                              neighborhood_name census_section_id                                                  geom
         17    Villaverde Villaverde alto, Casco Histórico de Villaverde             17120 SRID=4326;POLYGON((19637 2201,-6 207,19630 2408,30...
         17    Villaverde                                  San Cristóbal             17042 SRID=4326;POLYGON((20323 2875,-131 -1,20077 2892,1...
         17    Villaverde                                  San Cristóbal             17041 SRID=4326;POLYGON((20077 2892,-2 43,20076 2936,-84...


📋 TABLE: FACT_DEMOGRAPHICS
Columns (13 total):
  • census_section_id: VARCHAR
  • total_population: INTEGER
  • density: DOUBLE
  • edad_promedio: DOUBLE
  • propo

## 3. Simple Data Analysis

Now let's do some basic analysis with our data:

In [87]:
# Let's explore the geography and demographics data together
print("=== GEOGRAPHY + DEMOGRAPHICS ANALYSIS ===\n")

# Simple join to see districts and their demographics
district_summary = conn.execute("""
    SELECT 
        g.district_name,
        COUNT(*) as census_sections,
        ROUND(AVG(d.density), 1) as avg_density,
        ROUND(AVG(d.edad_promedio), 1) as avg_age,
        ROUND(AVG(d.proporcion_envejecimiento), 1) as avg_elderly_proportion
    FROM dim_geography g
    INNER JOIN fact_demographics d ON g.census_section_id = d.census_section_id
    WHERE g.district_name IS NOT NULL
    GROUP BY g.district_name
    ORDER BY census_sections DESC
""").fetchall()

# Convert to DataFrame for easier viewing
districts_df = pd.DataFrame(district_summary, 
                          columns=['district_name', 'census_sections', 'avg_density', 
                                 'avg_age', 'avg_elderly_proportion'])

print("Madrid Districts Overview:")
print(districts_df.to_string(index=False))

print(f"\n📊 Summary:")
print(f"  • Total districts: {len(districts_df)}")
print(f"  • Total census sections: {districts_df['census_sections'].sum():,}")
print(f"  • Average age across Madrid: {districts_df['avg_age'].mean():.1f} years")
print(f"  • Average elderly proportion: {districts_df['avg_elderly_proportion'].mean():.1f}%")

=== GEOGRAPHY + DEMOGRAPHICS ANALYSIS ===

Madrid Districts Overview:
        district_name  census_sections  avg_density  avg_age  avg_elderly_proportion
               Latina              196        331.8     46.7                    24.2
Fuencarral - El Pardo              185        307.0     45.4                    24.1
          Carabanchel              181        379.1     43.8                    18.8
   Puente de Vallecas              174        367.7     43.4                    18.1
        Ciudad Lineal              168        371.1     46.0                    22.6
            Salamanca              124        381.2     46.1                    24.0
            Hortaleza              123        210.3     44.6                    21.6
             Chamberí              121        411.8     46.1                    24.0
               Tetuán              117        402.4     44.2                    19.3
           Arganzuela              111        455.1     45.8                    

In [88]:
print("\n=== PLOTTING INFRASTRUCTURE ===\n")

# Counting non-null values for each column in the routing_infrastructure table
infrastructure_counts = conn.execute("""
    SELECT 
        COUNT(infra_id) AS infra_id,
        COUNT(osm_id) AS osm_id,
        COUNT(osm_type) AS osm_type,
        COUNT(wheelchair) AS wheelchair,
        COUNT(wheelchair_score) AS wheelchair_score,
        COUNT(highway) AS highway,
        COUNT(barrier) AS barrier,
        COUNT(kerb) AS kerb,
        COUNT(crossing) AS crossing,
        COUNT(tactile_paving) AS tactile_paving,
        COUNT(ramp) AS ramp,
        COUNT(surface) AS surface,
        COUNT(smoothness) AS smoothness,
        COUNT(incline) AS incline,
        COUNT(width) AS width,
        COUNT(name) AS name,
        COUNT(lat) AS lat,
        COUNT(lon) AS lon,
        COUNT(geom) AS geom,
        COUNT(data_type) AS data_type,
        COUNT(created_at) AS created_at
    FROM routing_infrastructure
""").fetchone()

# Displaying the counts in a pandas DataFrame for better readability
infrastructure_counts_df = pd.DataFrame([infrastructure_counts], columns=[
    'infra_id', 'osm_id', 'osm_type', 'wheelchair', 'wheelchair_score',
    'highway', 'barrier', 'kerb', 'crossing', 'tactile_paving', 'ramp',
    'surface', 'smoothness', 'incline', 'width', 'name', 'lat', 'lon', 'geom',
    'data_type', 'created_at'
])

print("Non-Null Value Counts for routing_infrastructure Table:")
print(infrastructure_counts_df.to_string(index=False))



=== PLOTTING INFRASTRUCTURE ===

Non-Null Value Counts for routing_infrastructure Table:
 infra_id  osm_id  osm_type  wheelchair  wheelchair_score  highway  barrier  kerb  crossing  tactile_paving  ramp  surface  smoothness  incline  width  name   lat   lon  geom  data_type  created_at
    55703   55703     55703       55703             55703    55703    55703 55703     55703           55703 55703    55703       55703    55703      0 55703 55703 55703 55703      55703       55703


In [89]:
print("\n=== MOST COMMON VALUES IN INFRASTRUCTURE TABLE ===\n")

# Get the column names from the routing_infrastructure table
columns = [col[0] for col in conn.execute(f"DESCRIBE routing_infrastructure").fetchall()]

# Iterate over each column and find the most common values
for column in columns:
    try:
        # Execute a query to find the most frequent values and their counts
        most_common = conn.execute(f"""
            SELECT {column}, COUNT(*) AS count
            FROM routing_infrastructure
            GROUP BY {column}
            ORDER BY count DESC
            LIMIT 5
        """).fetchall()
        
        # Print the column name and its most common values
        if most_common:
            print(f"Column '{column}':")
            for value, count in most_common:
                print(f"  Value = '{value}' (Count: {count})")
        else:
            print(f"Column '{column}': No data available")
            
    except Exception as e:
        print(f"Error processing column '{column}': {e}")
    
    print("-" * 40)



=== MOST COMMON VALUES IN INFRASTRUCTURE TABLE ===

Column 'infra_id':
  Value = '1' (Count: 1)
  Value = '12' (Count: 1)
  Value = '17' (Count: 1)
  Value = '27' (Count: 1)
  Value = '56' (Count: 1)
----------------------------------------
Column 'osm_id':
  Value = '20952908' (Count: 1)
  Value = '20960701' (Count: 1)
  Value = '21051126' (Count: 1)
  Value = '21716565' (Count: 1)
  Value = '22525252' (Count: 1)
----------------------------------------
Column 'osm_type':
  Value = 'node' (Count: 55703)
----------------------------------------
Column 'wheelchair':
  Value = 'unknown' (Count: 52564)
  Value = 'yes' (Count: 2934)
  Value = 'no' (Count: 175)
  Value = 'limited' (Count: 28)
  Value = 'designated' (Count: 2)
----------------------------------------
Column 'wheelchair_score':
  Value = '0' (Count: 52566)
  Value = '3' (Count: 2934)
  Value = '1' (Count: 175)
  Value = '2' (Count: 28)
----------------------------------------
Column 'highway':
  Value = 'crossing' (Count: 47

In [83]:
conn.close()